# Ensemble models

**One idea behind every model in this notebook:** one model makes mistakes. A *team* of models that make *different* mistakes, combined well, makes fewer.

Think of asking six friends for directions instead of one. If each friend is right most of the time, and they don't all copy the same map, the majority answer is right more often than any one friend.

There are three ways to build the team:

| Family | How the team is built | Models in this notebook |
|---|---|---|
| **Bagging** | Many models trained **side by side**, each on a random sample of the rows. Equal vote. | `BaggingClassifier`, `RandomForestClassifier` |
| **Boosting** | Models trained **one after another**, each one fixing the last one's mistakes. | `AdaBoostClassifier`, `GradientBoostingClassifier` |
| **Stacking** | Different kinds of model, plus a "manager" model that learns whose answer to trust. | `StackingClassifier` |

**Data:** the penguins file, with two columns (bill length and flipper length) used to predict the species (Adelie, Chinstrap or Gentoo).

## Step 1: load the data and split it

- `dropna()` removes penguins with missing values.
- `test_size=0.25` keeps a quarter of the penguins hidden for the test.
- `stratify=y` keeps the same mix of species in both parts.
- `random_state=42` makes the split the same every time we run it.

In [11]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

penguins = pd.read_csv("../../../data/penguins_size.csv")
penguins = penguins.dropna()

X = penguins[["culmen_length_mm", "flipper_length_mm"]]
y = penguins["species"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=42)

print("Penguins to learn from:", len(X_train))
print("Penguins kept for the test:", len(X_test))

Penguins to learn from: 250
Penguins kept for the test: 84


## Step 2: one tree vs a random forest

**Decision tree:** asks yes/no questions about the columns ("Is the flipper longer than 207 mm?") until it reaches an answer.

**Random forest:** many trees that vote. Two kinds of randomness make the trees different from each other:
1. Each tree learns from a **random sample of the rows**, drawn *with replacement*, so some penguins appear twice and some not at all. This is called a **bootstrap sample**.
2. At **every question**, each tree may only look at a **random few columns**.

**`max_depth`** = the most questions a tree may ask in a row before answering.
- Small (1, 2, 3): simple trees that may **underfit** (miss real patterns).
- `None` (the default): the tree keeps asking until every training penguin is sorted. It may **overfit** (memorise the training data).

In the cell below, both models use `max_depth=2`.

In [4]:
tree = DecisionTreeClassifier(max_depth=2, random_state=42)
tree.fit(X_train, y_train)

forest = RandomForestClassifier(max_depth=2, random_state=42)
forest.fit(X_train, y_train)

print("Tree   train:", round(accuracy_score(y_train, tree.predict(X_train)), 3))
print("Tree   test: ", round(accuracy_score(y_test, tree.predict(X_test)), 3))
print("Forest train:", round(accuracy_score(y_train, forest.predict(X_train)), 3))
print("Forest test: ", round(accuracy_score(y_test, forest.predict(X_test)), 3))

Tree   train: 0.952
Tree   test:  0.94
Forest train: 0.948
Forest test:  0.917


In [5]:
for depth in [1, 2, 3, 4, None]:
    tree = DecisionTreeClassifier(max_depth=depth, random_state=42)
    tree.fit(X_train, y_train)

    forest = RandomForestClassifier(max_depth=depth, random_state=42)
    forest.fit(X_train, y_train)

    tree_test = round(accuracy_score(y_test, tree.predict(X_test)), 3)
    forest_test = round(accuracy_score(y_test, forest.predict(X_test)), 3)
    print("max_depth", depth, "  Tree test:", tree_test, "  Forest test:", forest_test)

max_depth 1   Tree test: 0.774   Forest test: 0.774
max_depth 2   Tree test: 0.94   Forest test: 0.917
max_depth 3   Tree test: 0.94   Forest test: 0.929
max_depth 4   Tree test: 0.952   Forest test: 0.952
max_depth None   Tree test: 0.905   Forest test: 0.952


**Reading the result:** here the forest (0.917) does *worse* than the single tree (0.94). Why?

- A forest fixes **overfitting** (trees that are too wobbly), not **underfitting** (trees that are too simple).
- With `max_depth=2` every tree is short and simple. Averaging 100 simple trees still gives a simple answer.
- On top of that, each forest tree sometimes cannot see the best column (randomness 2 above), which makes short trees even weaker.

Same split, other depths (test accuracy):

| `max_depth` | 1 | 2 | 3 | 4 | `None` (default) |
|---|---|---|---|---|---|
| Tree | 0.774 | 0.94 | 0.94 | 0.952 | 0.905 |
| Forest | 0.774 | 0.917 | 0.929 | 0.952 | 0.952 |

With no depth limit, the single tree memorises the training data (train 1.0, test 0.905). The forest also scores 1.0 on train but holds 0.952 on test, because the vote cancels out each tree's random mistakes.

**Rule:** a forest works best with **deep** trees. Leave `max_depth=None` and let the vote do the work.

## Step 3: bagging

**Bagging** is short for **b**ootstrap **agg**regat**ing**:
1. **Bootstrap:** make many random samples of the rows (with replacement). 
2. **Aggregating:** train one model per sample, then let them **vote**.

### `BaggingClassifier` vs `RandomForestClassifier`

| | `BaggingClassifier()` | `RandomForestClassifier()` |
|---|---|---|
| Rows each model sees | Random sample with replacement | Same |
| Columns at each question | **All** of them | A **random few** |
| What it can wrap | **Any** model (tree, K-NN, logistic regression...) | Decision trees only |
| Do the trees look alike? | Fairly. If one column is very strong, every tree asks about it first | Less. Some trees can't see the strong column and find other routes |

**Why the forest adds column randomness:** voting only helps if the voters make **different** mistakes. Ten experts who read the same report make the same mistakes. A random forest is **bagging with trees, plus a random choice of columns at every question.**

With only 2 columns, the forest picks 1 of the 2 at each question (the square root of 2, rounded down).

### Hyperparameters

| Setting | Default | Meaning |
|---|---|---|
| `estimator` | a decision tree | The model to copy many times |
| `n_estimators` | 10 | How many copies (models in the team) |
| `max_samples` | 1.0 | Size of each sample, as a share of the rows (1.0 = same size as the data) |
| `max_features` | 1.0 | Share of columns each model gets (chosen once per model, not per question) |
| `bootstrap` | `True` | Draw rows with replacement |

In [3]:
from sklearn.ensemble import BaggingClassifier

bagging = BaggingClassifier(estimator=DecisionTreeClassifier(), n_estimators=100, random_state=42)
bagging.fit(X_train, y_train)

print("Bagging train:", round(accuracy_score(y_train, bagging.predict(X_train)), 3))
print("Bagging test: ", round(accuracy_score(y_test, bagging.predict(X_test)), 3))

Bagging train: 1.0
Bagging test:  0.964


**Reading the result:** bagging 100 full-depth trees gives train 1.0 and test 0.964. Each tree memorises its own sample, but their mistakes are different, so the vote cancels many of them out.

With the default `n_estimators=10` the same code scores 0.952. More trees made the vote steadier.

On this small, easy data bagging (0.964) scores a little above the default forest (0.952). That is normal: the forest's advantage shows with **many** columns, where one or two strong columns would otherwise take over every tree.

In [12]:
from sklearn.ensemble import BaggingClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB

models = {
    "Decision tree": DecisionTreeClassifier(random_state=42),
    "K-NN": KNeighborsClassifier(),
    "Logistic regression": LogisticRegression(max_iter=1000),
    "Naive Bayes": GaussianNB(),
}

for name, model in models.items():
    model.fit(X_train, y_train)
    alone = round(accuracy_score(y_test, model.predict(X_test)), 3)

    bagging = BaggingClassifier(estimator=model, n_estimators=100, random_state=42)
    bagging.fit(X_train, y_train)
    bagged = round(accuracy_score(y_test, bagging.predict(X_test)), 3)

    print(name, "  alone:", alone, "  bagged:", bagged)

Decision tree   alone: 0.905   bagged: 0.964
K-NN   alone: 0.976   bagged: 0.976
Logistic regression   alone: 0.964   bagged: 0.964
Naive Bayes   alone: 0.964   bagged: 0.964


## Step 4: bagging doesn't help every model

Bagging helps models that are **unstable**: change a few training rows and the model changes a lot. Deep decision trees are like that.

**K-NN** (k nearest neighbours) is **stable**: it answers by looking at the 5 closest penguins. Removing or repeating a few penguins barely changes who the neighbours are.

**Guess first:** will 100 bagged K-NN models beat one K-NN?

In [7]:
from sklearn.neighbors import KNeighborsClassifier

knn = KNeighborsClassifier()
knn.fit(X_train, y_train)

bagged_knn = BaggingClassifier(estimator=KNeighborsClassifier(), n_estimators=100, random_state=42)
bagged_knn.fit(X_train, y_train)

print("One K-NN:        ", round(accuracy_score(y_test, knn.predict(X_test)), 3))
print("100 bagged K-NN: ", round(accuracy_score(y_test, bagged_knn.predict(X_test)), 3))

One K-NN:         0.976
100 bagged K-NN:  0.976


**Reading the result:** both score 0.976. All 100 K-NN models give nearly the same answers, so the vote adds nothing.

**Rule:** bag models that are **wobbly** (deep trees). Don't bother bagging models that are already **steady** (K-NN, logistic regression).

## Step 5: AdaBoost (boosting)

Bagging builds its models **side by side**. **Boosting** builds them **one after another**, and each new model learns from the mistakes of the ones before.

**How AdaBoost works:**
1. Start with every penguin counting equally.
2. Train a **stump**: a tree that asks only **one** question (`max_depth=1`).
3. Make the penguins it got wrong **count more**, and the ones it got right count less.
4. Train the next stump on the reweighted penguins, so it pays most attention to the hard ones.
5. Repeat `n_estimators` times.
6. Every stump votes, but **not equally**: more accurate stumps get a bigger **say**. You can see these in `ada.estimator_weights_`.

### Hyperparameters

| Setting | Default | Meaning |
|---|---|---|
| `estimator` | a stump (`DecisionTreeClassifier(max_depth=1)`) | The weak model to repeat. The cell below writes it out, but it's already the default |
| `n_estimators` | 50 | How many stumps, built one after another |
| `learning_rate` | 1.0 | How big a say each stump gets. Smaller is slower and more careful, but needs more stumps |

**Guess first:** the cell tries 1, 2, 3, 10 and 50 stumps. Will accuracy go up every time?

In [5]:
from sklearn.ensemble import AdaBoostClassifier

for count in [1, 2, 3, 10, 50]:
    ada = AdaBoostClassifier(estimator=DecisionTreeClassifier(max_depth=1),
                             n_estimators=count, random_state=42)
    ada.fit(X_train, y_train)
    print(count, "stumps:  train", round(accuracy_score(y_train, ada.predict(X_train)), 3),
          "  test", round(accuracy_score(y_test, ada.predict(X_test)), 3))

1 stumps:  train 0.788   test 0.774
2 stumps:  train 0.604   test 0.595
3 stumps:  train 0.944   test 0.929
10 stumps:  train 0.948   test 0.94
50 stumps:  train 0.944   test 0.929


**Reading the result:**
- **1 stump (0.774):** one question can't separate three species.
- **2 stumps (0.595), worse than one!** Stump 1 asks about flipper length (≤ 207.5 mm). Stump 2 focuses on stump 1's mistakes (mostly Chinstraps) and asks about bill length (≤ 42.35 mm). Stump 2 gets a slightly bigger say (2.15 vs 2.01), so its vote wins, and stump 2 has never learned what a Gentoo looks like.
- **3 or more stumps (about 0.93):** the stumps now cover each other's gaps.
- **10 vs 50 (0.94 vs 0.929):** more stumps did not help. Boosting can start chasing noise.

**Try a smaller `learning_rate`** (same split): with `learning_rate=0.5` and 50 stumps, test accuracy is 0.952. Smaller steps, a steadier team.

## Step 6: gradient boosting

Also boosting, so trees are built **one after another**. The difference is **how** each new tree learns from the earlier mistakes:

- **AdaBoost:** makes the wrong rows *count more*, and the next stump still predicts the species.
- **Gradient boosting:** gives the next tree a **new target**: the **leftover error** of the team so far. Each tree learns "what is still missing".

**Everyday example (predicting a house price):**
1. First guess: the average price, \$600k.
2. One house really sold for \$750k, so \$150k is still missing.
3. Tree 1 learns to predict the missing part and says +\$120k.
4. The new guess is \$600k + 0.1 × \$120k. The `learning_rate` (0.1) means we only take a small step, so we don't overreact.
5. Tree 2 learns what is *still* missing. Repeat.

$$F_m(x) = F_{m-1}(x) + \gamma_m \, h_m(x)$$

- $F_{m-1}$ = the team so far
- $h_m$ = the new tree, trained on the leftover error
- $\gamma_m$ = how big a step to take with it

For classification it works the same way on probabilities instead of prices. With three species it builds **one small tree per species** each round.

### Hyperparameters

| Setting | Default | Meaning |
|---|---|---|
| `n_estimators` | 100 | How many rounds of trees |
| `learning_rate` | 0.1 | How much of each tree's correction we use. Smaller is slower but safer |
| `max_depth` | 3 | Size of each tree: small, but bigger than AdaBoost's stumps |
| `subsample` | 1.0 | Share of rows each tree sees. Below 1.0 adds randomness, like bagging |

In [8]:
from sklearn.ensemble import GradientBoostingClassifier

boosting = GradientBoostingClassifier(random_state=42)
boosting.fit(X_train, y_train)

print("Gradient boosting train:", round(accuracy_score(y_train, boosting.predict(X_train)), 3))
print("Gradient boosting test: ", round(accuracy_score(y_test, boosting.predict(X_test)), 3))

Gradient boosting train: 1.0
Gradient boosting test:  0.952


**Reading the result:** train 1.0, test 0.952. A perfect training score is a warning sign of overfitting.

What happens with other settings on the same split:

| `n_estimators` | `learning_rate` | Train | Test |
|---|---|---|---|
| 100 | 0.1 (default) | 1.0 | 0.952 |
| 500 | 0.1 | 1.0 | 0.94 |
| 100 | 0.02 | 0.984 | 0.964 |
| 500 | 0.02 | 1.0 | 0.929 |

- **More rounds can overfit:** 500 rounds scored lower than 100.
- **Smaller steps helped:** `learning_rate=0.02` with 100 rounds scored best (0.964).
- `n_estimators` and `learning_rate` work **together**: smaller steps need more rounds, but too many rounds overfit again.

## Step 7: stacking

Bagging and boosting build a team of the **same kind** of model. **Stacking** builds a team of **different kinds** of model, then adds a **manager**.

**How it works:**
1. **Team members:** here a decision tree and a K-NN. They look at penguins in different ways (yes/no questions vs nearest neighbours), so they make different mistakes.
2. Each member gives its answer as **probabilities**, e.g. "80% Chinstrap, 20% Adelie".
3. **Manager** (`final_estimator`): a logistic regression that takes the members' answers as its input columns and learns **whose answer to trust, and when**.

**Why `cv=5`?** The manager must learn from answers the members gave on penguins they had **not** trained on. Otherwise every member looks perfect on its own training data and the manager learns to trust whoever memorised best. `cv=5` splits the training data into 5 parts. Each member answers for one part after training on the other four.

### Hyperparameters

| Setting | Default | Meaning |
|---|---|---|
| `estimators` | (you must give it) | The team: a list of `("name", model)` pairs |
| `final_estimator` | logistic regression | The manager |
| `cv` | 5 | How many parts to split the training data into, so members answer on penguins they haven't seen |

**Guess first:** the tree scores 0.94 and K-NN 0.976 on their own. Will the stack beat both?

In [9]:
from sklearn.ensemble import StackingClassifier
from sklearn.linear_model import LogisticRegression

team = [
    ("tree", DecisionTreeClassifier(max_depth=3, random_state=42)),
    ("knn", KNeighborsClassifier()),
]

stack = StackingClassifier(estimators=team,
                           final_estimator=LogisticRegression(max_iter=1000),
                           cv=5)
stack.fit(X_train, y_train)

for name, model in team:
    model.fit(X_train, y_train)
    print(name, "test: ", round(accuracy_score(y_test, model.predict(X_test)), 3))
print("stack test:", round(accuracy_score(y_test, stack.predict(X_test)), 3))

tree test:  0.94
knn test:  0.976
stack test: 0.964


**What does the manager actually see?** The cell below shows the manager's input for the first test penguin: each member's probabilities for each species.

In [10]:
first_penguin = X_test.iloc[[0]]

column_names = []
for name, model in team:
    for species in stack.classes_:
        column_names.append(name + " says " + species)

manager_input = pd.DataFrame(stack.transform(first_penguin), columns=column_names)
print(manager_input.round(2).T)
print()
print("Manager's answer:", stack.predict(first_penguin)[0])
print("Real species:    ", y_test.iloc[0])

                       0
tree says Adelie     0.0
tree says Chinstrap  1.0
tree says Gentoo     0.0
knn says Adelie      0.2
knn says Chinstrap   0.8
knn says Gentoo      0.0

Manager's answer: Chinstrap
Real species:     Chinstrap


**Reading the result:**
- Tree 0.94, K-NN 0.976, stack 0.964: the stack lands **between** its two members. It does not beat the best one here.
- Stacking is **not guaranteed** to win. It helps most when members are good at **different** penguins. On this easy data K-NN is already right almost every time, so the manager has little to add.
- For the first test penguin the tree says 100% Chinstrap and K-NN says 80% Chinstrap, 20% Adelie. The manager reads all six numbers and answers Chinstrap, which is correct.

## Cheat sheet

### `n_estimators`: how many models are in the team

| Model | Default `n_estimators` |
|---|---|
| `BaggingClassifier` | 10 |
| `RandomForestClassifier` | 100 |
| `AdaBoostClassifier` | 50 |
| `GradientBoostingClassifier` | 100 |

- **Bagging and random forest (models built side by side):** more models **never makes things worse**. Accuracy rises, then levels off. Extra models only cost time. Pick a number large enough that accuracy stops rising (100 to 500).
- **Boosting (models built one after another):** each new model keeps fixing leftover mistakes, and eventually it starts fixing **noise**. Too many can **overfit**. Tune `n_estimators` **together with** `learning_rate`.

### All four models side by side

| | Bagging | Random forest | AdaBoost | Gradient boosting |
|---|---|---|---|---|
| Built | Side by side | Side by side | One after another | One after another |
| Each model | Deep tree (or any model) | Deep tree, random columns | Stump | Small tree (depth 3) |
| Next model learns from | Nothing (independent) | Nothing (independent) | Mistakes made to count more | Leftover error |
| Vote | Equal | Equal | Weighted by accuracy | Sum of small steps |
| Fixes | Overfitting | Overfitting | Underfitting | Underfitting |
| Too many models? | Harmless | Harmless | Can overfit | Can overfit |

### Hyperparameters to remember

| Hyperparameter | Used by | Plain meaning | Starting point |
|---|---|---|---|
| `n_estimators` | all four | How many models | Forest 100 to 500; boosting: tune |
| `max_depth` | forest, gradient boosting | Most questions per tree | Forest: `None`; gradient boosting: 3 |
| `max_features` | forest, bagging | Columns each tree may look at | Forest: `"sqrt"` |
| `learning_rate` | AdaBoost, gradient boosting | How big each correction step is | 0.1, smaller with more models |
| `estimator` | bagging, AdaBoost | Which model to repeat | Bagging: deep tree; AdaBoost: stump |
| `random_state` | all four | Same random choices each run | Any fixed number, e.g. 42 |

### In one sentence each

- **Bagging:** many strong models, each trained on a different random sample of rows, vote equally.
- **Random forest:** bagging with trees, plus a random choice of columns at every question.
- **AdaBoost:** many weak models, each trained to fix the last one's mistakes, vote by how good they are.
- **Gradient boosting:** each new tree learns the leftover error and adds a small correction.
- **Stacking:** different models answer, and a manager model learns whose answer to trust.

> AdaBoost shouts louder about the rows it got wrong. Gradient boosting tells the next tree exactly how far off it was.

**A caution:** these scores come from **one** split of 84 test penguins, where one penguin is worth about 0.012. Differences of 0.01 to 0.02 can be luck. Cross-validation (several splits, averaged) gives a fairer comparison.